# Initial denoising tests: Topaz-Denoise and DRACO

This notebook is a small-subset harness for comparing:

1. **Topaz-Denoise pretrained general model**
2. **Topaz-Denoise retraining** on this dataset, after paired noisy observations are prepared
3. **DRACO pretrained Denoise/B**

Important: Topaz-Denoise and DRACO outputs should initially be evaluated as visualization/picking aids. For reconstruction, prefer raw data unless a downstream workflow has been explicitly validated.

In [ ]:
from pathlib import Path
import os
import random
import shutil
import subprocess
import sys

PROJECT = Path('/nfs/bartesaghilab2/zy223/denoising')
SOURCE = Path('/nfs/bartesaghilab/micromon/research-bartesaghilab-06/shared/users/zy223@duke.edu/projects/Prestin-1fhwPcd8gQ/sp-preprocessing-ghLbktywzmKi35IA/mrc')
SUBSET = PROJECT / 'data/subset'
RESULTS = PROJECT / 'results'

for path in [SUBSET, RESULTS / 'topaz_pretrained', RESULTS / 'topaz_retrained', RESULTS / 'draco_pretrained']:
    path.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT)
print('Source exists:', SOURCE.exists())
print('Available MRC files:', len(list(SOURCE.glob('*.mrc'))))

## Environment checks

The base login environment may not have Topaz, DRACO, or `mrcfile`. Run this from the kernel you intend to use.

In [ ]:
def which(name):
    return shutil.which(name) or 'not found'

def has_module(name):
    try:
        __import__(name)
        return True
    except Exception:
        return False

print('topaz:', which('topaz'))
print('draco:', which('draco'), '(DRACO may be repo/notebook-based)')
for module in ['mrcfile', 'numpy', 'matplotlib', 'torch']:
    print(f'{module}:', has_module(module))

## Create or refresh a small subset

This uses symlinks to avoid copying large files. Increase `N_SUBSET` after the first test passes.

In [ ]:
N_SUBSET = 5
SEED = 20260526

cmd = [
    sys.executable, str(PROJECT / 'scripts/make_subset.py'),
    '--source', str(SOURCE),
    '--dest', str(SUBSET),
    '--n', str(N_SUBSET),
    '--seed', str(SEED),
    '--overwrite',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
subset_files = sorted(SUBSET.glob('*.mrc'))
subset_files

## Inspect MRC shape if `mrcfile` is available

This helps determine whether the files are frame stacks (3D array) or summed micrographs (2D array), which matters for Topaz retraining.

In [ ]:
if has_module('mrcfile'):
    import mrcfile
    for path in subset_files[:3]:
        with mrcfile.open(path, permissive=True) as mrc:
            print(path.name, mrc.data.shape, mrc.data.dtype, 'voxel_size=', mrc.voxel_size)
else:
    print('Install mrcfile in this kernel to inspect MRC dimensions.')

## Topaz-Denoise pretrained run

Topaz-Denoise pretrained models are the first test because they do not require paired even/odd training inputs. Verify CLI flags with `topaz denoise --help` in the Topaz environment before running.

In [ ]:
topaz_out = RESULTS / 'topaz_pretrained'
topaz_cmd = [
    'topaz', 'denoise',
    '--model', 'unet',
    '--device', '0',
    '--patch-size', '4000',
    '--patch-padding', '500',
    '--destdir', str(topaz_out),
    *map(str, subset_files),
]
print(' '.join(topaz_cmd))
if shutil.which('topaz'):
    print('Topaz is available. Uncomment the next line to run after checking `topaz denoise --help`.')
    # subprocess.run(topaz_cmd, check=True)
else:
    print('Topaz not found in this kernel environment.')

## Topaz-Denoise retraining plan

Retraining should wait until we confirm input type and produce paired noisy observations. In the Topaz-Denoise paper, pairs come from even/odd movie frames independently summed/aligned, then trained with Noise2Noise.

Checklist:

- Confirm `*_fractions.mrc` are frame stacks or locate raw movie frames.
- Build even/odd frame sums for each movie.
- Normalize/prepare paired file lists expected by the installed Topaz training command.
- Train on a small split first, validate visually, then scale.

## DRACO pretrained setup

The DRACO repo README points to notebooks/Hugging Face rather than a stable CLI. Clone the repo and install its environment, then adapt the DRACO denoising notebook/API to the `subset_files`. Remember model weights are non-commercial and should not be redistributed outside approved use.

In [ ]:
DRACO_REPO = PROJECT / 'external/draco'
print('Expected DRACO repo:', DRACO_REPO)
print('Exists:', DRACO_REPO.exists())
print('Subset files for DRACO:')
for path in subset_files:
    print(path)
print('DRACO output:', RESULTS / 'draco_pretrained')

## Visual comparison placeholder

After outputs are generated, load raw, Topaz, and DRACO MRCs here for side-by-side visualization and summary metrics. Keep this qualitative at first; denoising can alter apparent features.